In [1]:
import pandas as pd

In [17]:
df = pd.read_csv('../temp/demotermine_cleaned_2022_06_09_DE.csv', parse_dates=['date', 'edit_date'])

# only german

In [18]:
df = df[df['lang'] == 'de']

In [19]:
df.text_length.describe()

count    23929.000000
mean       284.992436
std        422.936376
min          5.000000
25%         84.000000
50%        141.000000
75%        314.000000
max       4221.000000
Name: text_length, dtype: float64

# min, max text length

In [20]:
df = df[(df['text_length'] >= 100) & (df['text_length'] <= 400)]

In [21]:
len(df)

11358

In [22]:
df.text_length.sum() // 500000

4

# select sample 

In [8]:
twenty_percent = int(len(df) / 100 * 20)

In [9]:
sample_df = df.sample(twenty_percent, random_state=42)

In [10]:
sample_df.head()

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,channel_name,channel_id,channel_description,message_id,from_id,via_bot_id,...,forwards,fwd_from,replies,reply_to,media,views,id,cleaned_text,lang,text_length
244,244,278,278,307,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,5172,NaN,NaN,...,18.0,NaN,"MessageReplies(replies=0, replies_pts=290300, ...",NaN,MessageMediaPhoto(photo=Photo(id=5474467780972...,3542.0,Demotermine125028861051722021-01-15 00:45:39,Nürnberg Richard-Wagner-Platz ( Sterntor ) Vid...,de,150
16913,16913,19785,19785,22034,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,25582,NaN,NaN,...,7.0,NaN,"MessageReplies(replies=0, replies_pts=290292, ...",NaN,MessageMediaDocument(document=Document(id=6066...,1328.0,Demotermine1250288610255822021-11-07 07:07:40,WELL DONE VICTORIA ! YOU ARE NOT ALONE ... W...,de,144
21961,21961,25499,25499,28539,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,1621,NaN,NaN,...,4.0,NaN,NaN,NaN,MessageMediaDocument(document=Document(id=5442...,4751.0,Demotermine125028861016212020-06-22 14:37:40,"Hallöchen , liebe Grüße aus Bad Tabarz wir tre...",de,226
18082,18082,21337,21337,23879,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,23532,NaN,NaN,...,4.0,NaN,"MessageReplies(replies=0, replies_pts=290292, ...",NaN,MessageMediaPhoto(photo=Photo(id=5458820777026...,1758.0,Demotermine1250288610235322021-10-27 05:20:30,"SAFE - THE - DATES : - # mahnwache , Darmstadt...",de,315
5734,5734,6828,6828,7913,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,21266,NaN,NaN,...,3.0,NaN,"MessageReplies(replies=0, replies_pts=290293, ...",NaN,MessageMediaDocument(document=Document(id=5996...,1649.0,Demotermine1250288610212662021-10-13 20:51:46,"Wegweisung für den mornigen Tag , 14. Oktober ...",de,224


In [11]:
sample_df.text_length.sum()

425613

In [12]:
sample_df.to_csv('../temp/sample_data_2022_06_09.csv')

In [14]:
len(df)

11358

## Identify similar messages via Levenshtein Distance

In [13]:
from itertools import combinations
import Levenshtein as lev

In [23]:
# df = sample_df.copy()

In [24]:
len(df)

11358

In [25]:
lev_df = pd.DataFrame() 
new_df = pd.DataFrame(combinations(df['cleaned_text'],2), columns=["text_1", "text_2"]) 
new_df["lev_score"] = new_df.apply(lambda x: lev.ratio(x[0],x[1]), axis=1) 
lev_df = pd.concat([lev_df, new_df], axis=0).reset_index(drop=True)

In [26]:
# Get message ids
lev_df_ids = pd.DataFrame() 
new_df_ids = pd.DataFrame(combinations(df['message_id'],2), columns=["message_id_1", "message_id_2"]) 
lev_df_ids = pd.concat([lev_df_ids, new_df_ids], axis=0).reset_index(drop=True)

In [27]:
# combine lev results with message ids 
levs = pd.DataFrame()
levs = pd.concat([lev_df,lev_df_ids], axis=1)

In [28]:
levs.head()

,text_1,text_2,lev_score,message_id_1,message_id_2
0,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,27 Feb : Sternmarsch – Graz BEHÖRDLICH VERB...,0.334247,15971,6885
1,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,Stuttgart-Stream von Stephan Bergmann u.a. mit...,0.312102,15971,4836
2,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,Demo # MobilitätswendeJetzt – Lobauautobahn ...,0.398754,15971,11105
3,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,Das kommende Desaster : Rot-Rot-Grün mit Schol...,0.289941,15971,17547
4,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,... eine gute Zusammenfassung aus Zermatt ... ...,0.351097,15971,24799


In [29]:
# get ids of messages over certain threshold
second_ids = levs[levs['lev_score'] >= 0.8].message_id_2.tolist()

In [30]:
len(set(second_ids))

3266

In [31]:
ids_to_drop = list(set(second_ids))

In [32]:
result_df = df[~df.message_id.isin(ids_to_drop)]

In [44]:
len(result_df)

8092

In [45]:
result_df.to_csv('../temp/data_selection_2022_06_09-3_after_levenshtein.csv')

In [46]:
result_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 8092 entries, 0 to 23926
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            8092 non-null   object        
 1   message_id    8092 non-null   int64         
 2   date          8092 non-null   datetime64[ns]
 3   cleaned_text  8092 non-null   object        
 4   lang          8092 non-null   object        
 5   text_length   8092 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 442.5+ KB


In [47]:
columns_to_keep = ['id', 'message_id', 'date', 'cleaned_text', 'lang', 'text_length']

In [48]:
result_df = result_df[columns_to_keep]

In [49]:
result_df.head(20)

,id,message_id,date,cleaned_text,lang,text_length
0,Demotermine1250288610159712021-08-29 11:10:12,15971,2021-08-29 11:11:25,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,de,213
1,Demotermine125028861068852021-02-27 07:11:32,6885,2021-02-27 07:11:14,27 Feb : Sternmarsch – Graz BEHÖRDLICH VERB...,de,152
3,Demotermine125028861048362021-01-01 03:57:43,4836,2021-01-01 03:50:02,Stuttgart-Stream von Stephan Bergmann u.a. mit...,de,101
9,Demotermine1250288610111052021-05-27 07:00:29,11105,2021-05-27 07:00:28,Demo # MobilitätswendeJetzt – Lobauautobahn ...,de,108
11,Demotermine1250288610175472021-09-09 16:32:09,17547,2021-09-09 16:32:54,Das kommende Desaster : Rot-Rot-Grün mit Schol...,de,125
14,Demotermine1250288610247992021-11-03 08:22:51,24799,2021-11-03 08:22:34,... eine gute Zusammenfassung aus Zermatt ... ...,de,106
17,Demotermine1250288610212702021-10-13 21:09:17,21270,2021-10-13 21:09:25,Diesem Antrag kann jeder Wahlberechtigter Bürg...,de,300
20,Demotermine125028861046642020-12-21 11:26:06,4664,2020-12-21 11:23:20,Berlin Autokorso Treffpunkt Masurenallee P1 Ha...,de,184
21,Demotermine1250288610249102021-11-03 14:16:05,24910,2021-11-03 14:14:49,"Donnerstag , 4.11 11.30 Rathausplatz Erlange...",de,346
24,Demotermine1250288610154372021-08-28 15:05:28,15437,2021-08-28 15:05:53,+ + + Potsdamer Platz : Polizei sagt mit sympa...,de,383


In [50]:
result_df.text_length.sum()

1623544

In [51]:
len(result_df)

8092

In [42]:
result_df.to_csv('../temp/data_selection_final_2022_06_09-3.csv')

In [43]:
texts = result_df.cleaned_text.values

In [62]:
#for t in texts: 
 #   print(t, '\n\n')